# Attention-Based MIL Models Tutorial

This notebook demonstrates how to use attention-based Multiple Instance Learning (MIL) models for whole slide image (WSI) analysis in computational pathology.

## Overview

We'll cover:
1. Loading pre-trained attention models
2. Running inference on test slides
3. Extracting and visualizing attention weights
4. Comparing attention patterns across different models
5. Interpreting attention heatmaps for tumor detection

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h5py

from src.models.attention_mil import create_attention_model, AttentionMIL, CLAM, TransMIL
from src.utils.attention_utils import save_attention_weights, load_attention_weights
from src.visualization.attention_heatmap import AttentionHeatmapGenerator
from src.data.camelyon_slide_dataset import CAMELYONSlideDataset

## 1. Load Pre-trained Attention Model

First, let's load a pre-trained attention model. We'll use AttentionMIL as an example.

In [ ]:
# Configuration for AttentionMIL
config = {
    'model': {
        'wsi': {
            'model_type': 'attention_mil',
            'feature_dim': 1024,
            'hidden_dim': 256,
            'num_classes': 2,
            'dropout': 0.25,
            'attention_mil': {
                'gated': True,
                'attention_mode': 'instance'
            }
        }
    }
}

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = create_attention_model(config)
model = model.to(device)
model.eval()

print(f"Model loaded on {device}")
print(f"Model type: {type(model).__name__}")

### Load checkpoint (if available)

In [ ]:
# Load pre-trained weights
checkpoint_path = Path('checkpoints/attention_mil/best_model.pth')

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    print(f"Best validation AUC: {checkpoint['best_val_auc']:.4f}")
else:
    print("No checkpoint found, using randomly initialized model")

## 2. Load Test Data

Load the CAMELYON dataset for inference.

In [ ]:
# Load test dataset
data_dir = Path('data/camelyon16')
test_dataset = CAMELYONSlideDataset(
    data_dir=data_dir,
    split='test',
    max_patches=1000
)

print(f"Test dataset size: {len(test_dataset)} slides")

# Create data loader
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

## 3. Run Inference and Extract Attention Weights

Run the model on test slides and extract attention weights.

In [ ]:
# Run inference on a single slide
sample_batch = next(iter(test_loader))
features = sample_batch['features'].to(device)
num_patches = sample_batch['num_patches']
slide_id = sample_batch['slide_id'][0]
label = sample_batch['label'].item()

print(f"Slide ID: {slide_id}")
print(f"True label: {'Tumor' if label == 1 else 'Normal'}")
print(f"Number of patches: {num_patches.item()}")

# Forward pass with attention extraction
with torch.no_grad():
    logits, attention_weights = model(features, num_patches, return_attention=True)
    probs = torch.softmax(logits, dim=1)
    pred_label = torch.argmax(probs, dim=1).item()
    confidence = probs[0, pred_label].item()

print(f"Predicted label: {'Tumor' if pred_label == 1 else 'Normal'}")
print(f"Confidence: {confidence:.4f}")
print(f"Attention weights shape: {attention_weights.shape}")

### Analyze attention distribution

In [ ]:
# Get valid attention weights (exclude padding)
valid_attention = attention_weights[0, :num_patches.item()].cpu().numpy()

print(f"Attention statistics:")
print(f"  Mean: {valid_attention.mean():.6f}")
print(f"  Std: {valid_attention.std():.6f}")
print(f"  Min: {valid_attention.min():.6f}")
print(f"  Max: {valid_attention.max():.6f}")
print(f"  Sum: {valid_attention.sum():.6f} (should be ~1.0)")

# Plot attention distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(valid_attention, bins=50, edgecolor='black')
plt.xlabel('Attention Weight')
plt.ylabel('Frequency')
plt.title('Attention Weight Distribution')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(sorted(valid_attention, reverse=True))
plt.xlabel('Patch Rank')
plt.ylabel('Attention Weight')
plt.title('Attention Weights (Sorted)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Save and Load Attention Weights

In [ ]:
# Save attention weights
output_dir = Path('outputs/attention_weights')
output_dir.mkdir(parents=True, exist_ok=True)

# Get coordinates (if available in dataset)
if 'coordinates' in sample_batch:
    coordinates = sample_batch['coordinates'][0, :num_patches.item()].cpu().numpy()
else:
    # Create dummy coordinates for demonstration
    coordinates = np.random.randint(0, 10000, size=(num_patches.item(), 2))

save_attention_weights(
    attention_weights=valid_attention,
    coordinates=coordinates,
    slide_id=slide_id,
    output_dir=output_dir
)

print(f"Attention weights saved to {output_dir / slide_id}.h5")

In [ ]:
# Load attention weights
loaded_attention, loaded_coords = load_attention_weights(slide_id, output_dir)

if loaded_attention is not None:
    print(f"Loaded attention weights shape: {loaded_attention.shape}")
    print(f"Loaded coordinates shape: {loaded_coords.shape}")
    print(f"Values match: {np.allclose(loaded_attention, valid_attention)}")

## 5. Generate Attention Heatmaps

Visualize attention weights as heatmaps overlaid on slide thumbnails.

In [ ]:
# Create heatmap generator
heatmap_output_dir = Path('outputs/attention_heatmaps')
heatmap_generator = AttentionHeatmapGenerator(
    attention_dir=output_dir,
    output_dir=heatmap_output_dir,
    colormap='jet',
    thumbnail_size=(512, 512)
)

# Generate heatmap for the slide
heatmap_path = heatmap_generator.generate_heatmap(
    slide_id=slide_id,
    thumbnail_path=None  # Set to actual thumbnail path if available
)

print(f"Heatmap saved to {heatmap_path}")

In [ ]:
# Display the heatmap
from PIL import Image

if heatmap_path and Path(heatmap_path).exists():
    img = Image.open(heatmap_path)
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Attention Heatmap for {slide_id}')
    plt.show()

## 6. Compare Different Attention Models

Compare attention patterns from AttentionMIL, CLAM, and TransMIL.

In [ ]:
# Create different model types
model_configs = {
    'AttentionMIL': {
        'model_type': 'attention_mil',
        'attention_mil': {'gated': True, 'attention_mode': 'instance'}
    },
    'CLAM': {
        'model_type': 'clam',
        'clam': {'num_clusters': 10, 'multi_branch': True}
    },
    'TransMIL': {
        'model_type': 'transmil',
        'transmil': {'num_layers': 2, 'num_heads': 8, 'use_pos_encoding': True}
    }
}

# Run inference with each model
results = {}

for model_name, model_config in model_configs.items():
    # Create model
    config['model']['wsi'].update(model_config)
    model = create_attention_model(config).to(device)
    model.eval()
    
    # Run inference
    with torch.no_grad():
        logits, attention = model(features, num_patches, return_attention=True)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        conf = probs[0, pred].item()
    
    results[model_name] = {
        'prediction': pred,
        'confidence': conf,
        'attention': attention[0, :num_patches.item()].cpu().numpy()
    }
    
    print(f"{model_name}: Pred={'Tumor' if pred==1 else 'Normal'}, Conf={conf:.4f}")

In [ ]:
# Compare attention distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (model_name, result) in enumerate(results.items()):
    attention = result['attention']
    axes[idx].hist(attention, bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_xlabel('Attention Weight')
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'{model_name}\nConf: {result["confidence"]:.4f}')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Interpret Attention for Tumor Detection

Analyze which patches receive high attention and their correlation with tumor regions.

In [ ]:
# Get top-k patches by attention weight
k = 10
attention = results['AttentionMIL']['attention']
top_k_indices = np.argsort(attention)[-k:][::-1]

print(f"Top {k} patches by attention weight:")
print(f"{'Rank':<6} {'Index':<8} {'Attention':<12} {'Coordinates'}")
print("-" * 50)

for rank, idx in enumerate(top_k_indices, 1):
    attn_val = attention[idx]
    coord = coordinates[idx] if idx < len(coordinates) else [0, 0]
    print(f"{rank:<6} {idx:<8} {attn_val:<12.6f} ({coord[0]}, {coord[1]})")

In [ ]:
# Analyze attention concentration
# High concentration suggests model focuses on specific regions (e.g., tumor)
# Low concentration suggests model considers entire slide

def attention_entropy(attention_weights):
    """Calculate entropy of attention distribution."""
    # Add small epsilon to avoid log(0)
    eps = 1e-10
    attention_weights = attention_weights + eps
    entropy = -np.sum(attention_weights * np.log(attention_weights))
    return entropy

def attention_gini(attention_weights):
    """Calculate Gini coefficient of attention distribution."""
    sorted_weights = np.sort(attention_weights)
    n = len(sorted_weights)
    index = np.arange(1, n + 1)
    gini = (2 * np.sum(index * sorted_weights)) / (n * np.sum(sorted_weights)) - (n + 1) / n
    return gini

print("Attention concentration metrics:")
print(f"{'Model':<15} {'Entropy':<12} {'Gini':<12} {'Top-10%'}")
print("-" * 55)

for model_name, result in results.items():
    attention = result['attention']
    entropy = attention_entropy(attention)
    gini = attention_gini(attention)
    top_10_pct = np.sum(np.sort(attention)[-int(len(attention)*0.1):])
    
    print(f"{model_name:<15} {entropy:<12.4f} {gini:<12.4f} {top_10_pct:.4f}")

print("\nInterpretation:")
print("- Lower entropy = more concentrated attention")
print("- Higher Gini = more unequal distribution (focused)")
print("- Higher Top-10% = model focuses on small subset of patches")

## 8. Batch Processing

Process multiple slides and generate heatmaps in batch.

In [ ]:
# Process first 5 test slides
num_slides = min(5, len(test_dataset))
slide_ids = []

model = create_attention_model(config).to(device)
model.eval()

for i in range(num_slides):
    batch = test_dataset[i]
    features = batch['features'].unsqueeze(0).to(device)
    num_patches = torch.tensor([batch['num_patches']])
    slide_id = batch['slide_id']
    
    # Run inference
    with torch.no_grad():
        logits, attention = model(features, num_patches, return_attention=True)
    
    # Save attention weights
    valid_attention = attention[0, :num_patches.item()].cpu().numpy()
    coords = np.random.randint(0, 10000, size=(num_patches.item(), 2))  # Dummy coords
    
    save_attention_weights(
        attention_weights=valid_attention,
        coordinates=coords,
        slide_id=slide_id,
        output_dir=output_dir
    )
    
    slide_ids.append(slide_id)
    print(f"Processed {i+1}/{num_slides}: {slide_id}")

# Generate heatmaps in batch
heatmap_paths = heatmap_generator.generate_batch(slide_ids)
print(f"\nGenerated {len(heatmap_paths)} heatmaps")

## Summary

This tutorial demonstrated:
1. ✅ Loading pre-trained attention models (AttentionMIL, CLAM, TransMIL)
2. ✅ Running inference and extracting attention weights
3. ✅ Saving and loading attention weights to/from HDF5
4. ✅ Generating attention heatmaps for visualization
5. ✅ Comparing attention patterns across different models
6. ✅ Interpreting attention for tumor detection
7. ✅ Batch processing multiple slides

### Key Takeaways

- **AttentionMIL**: Simple gated attention, good baseline
- **CLAM**: Clustering-constrained attention, handles heterogeneous slides
- **TransMIL**: Transformer-based, captures long-range dependencies

### Next Steps

- Train models on your own dataset
- Experiment with different hyperparameters
- Compare performance on different tasks (classification, survival)
- Analyze attention patterns for clinical insights